In [1]:
import os
import pandas as pd
import numpy as np
import re

In [2]:
preprocessed_model_fit_folder = f'/Volumes/project/3025011.02/pre-processed/modelling'
model_fit_pattern = re.compile(r'^model-fit_sub-\d{3}_ses-\d{2}\.csv$')

In [3]:
model_fit_combined = []
# Iterate through files in the folder
for file_name in os.listdir(preprocessed_model_fit_folder):
    if model_fit_pattern.match(file_name):
        path = os.path.join(preprocessed_model_fit_folder, file_name)
        model_fit = pd.read_csv(path)
        model_fit_combined.append(model_fit)

model_fit_combined = pd.concat(model_fit_combined, ignore_index=True)
print(model_fit_combined)

   subject   model0_ll  model0_q_happy_pull  model0_q_happy_push  \
0        5 -433.348327             0.341634             0.587877   
1       10 -454.762216             0.548885            -0.701981   
2       14 -447.990521            -0.178037             0.023507   

   model0_q_angry_pull  model0_q_angry_push  model0_beta  model0_bias  \
0             0.754106             0.507421    18.489874     4.984568   
1            -0.900423             0.320666     2.484186    -3.192284   
2             0.403285             0.167132     6.800463     1.514143   

    model1_ll  model1_beta  ...  model3_lambda  model3_weight1  \
0 -234.252349     1.961054  ...       0.997527        0.629801   
1 -259.721480     1.428777  ...       0.996013        0.912852   
2 -388.594925     0.665279  ...       0.002473        0.407989   

   model3_weight2  model3_kappa  model3_bias   model4_ll  model4_beta  \
0        0.895605      0.557913     0.875518 -228.776161     2.014596   
1        0.796034      

In [4]:
# Automatically detect models
models = [col.split('_')[0] for col in model_fit_combined.columns if '_ll' in col]
models = sorted(set(models))

In [5]:
# Define number of trials per subject (why is this hardcoded?)
n_trials = 667
# Replace this by a column in the dataframe

In [6]:
# Initialize dictionaries for AIC/BIC
for model in models:
    # Count the number of parameters per model
    param_cols = [col for col in model_fit_combined.columns if col.startswith(model + '_') and col != model + '_ll']
    k = len(param_cols)

    # Get negative log-likelihoods
    ll = model_fit_combined[f'{model}_ll']
    #ll = -neg_ll  # Convert to positive log-likelihood

    # Calculate AIC and BIC
    model_fit_combined[f'{model}_AIC'] = 2 * k - 2 * ll
    model_fit_combined[f'{model}_BIC'] = k * np.log(n_trials) - 2 * ll

In [7]:
# Example: show AIC/BIC differences between model1 and model2
model_fit_combined['delta_AIC_model1_vs_model0'] = model_fit_combined['model1_AIC'] - model_fit_combined['model0_AIC']
model_fit_combined['delta_AIC_model2_vs_model0'] = model_fit_combined['model2_AIC'] - model_fit_combined['model0_AIC']
model_fit_combined['delta_AIC_model2_vs_model1'] = model_fit_combined['model2_AIC'] - model_fit_combined['model1_AIC']
model_fit_combined['delta_AIC_model3_vs_model0'] = model_fit_combined['model3_AIC'] - model_fit_combined['model0_AIC']
model_fit_combined['delta_AIC_model3_vs_model1'] = model_fit_combined['model3_AIC'] - model_fit_combined['model1_AIC']
model_fit_combined['delta_AIC_model3_vs_model2'] = model_fit_combined['model3_AIC'] - model_fit_combined['model2_AIC']
model_fit_combined['delta_BIC_model1_vs_model0'] = model_fit_combined['model1_BIC'] - model_fit_combined['model0_BIC']
model_fit_combined['delta_BIC_model2_vs_model0'] = model_fit_combined['model2_BIC'] - model_fit_combined['model0_BIC']
model_fit_combined['delta_BIC_model2_vs_model1'] = model_fit_combined['model2_BIC'] - model_fit_combined['model1_BIC']
model_fit_combined['delta_BIC_model3_vs_model0'] = model_fit_combined['model3_BIC'] - model_fit_combined['model0_BIC']
model_fit_combined['delta_BIC_model3_vs_model1'] = model_fit_combined['model3_BIC'] - model_fit_combined['model1_BIC']
model_fit_combined['delta_BIC_model3_vs_model2'] = model_fit_combined['model3_BIC'] - model_fit_combined['model2_BIC']

In [8]:
# Save or view
print(model_fit_combined[['subject'] + [col for col in model_fit_combined.columns if 'AIC' in col or 'BIC' in col]])
model_fit_combined.to_csv(f'{preprocessed_model_fit_folder}/model_fit_comparison.csv', index=False)

   subject  model0_AIC  model0_BIC  model1_AIC  model1_BIC  model2_AIC  \
0        5  878.696654  905.713394  474.504698  488.013069  468.421886   
1       10  921.524432  948.541173  525.442959  538.951329  493.455491   
2       14  907.981043  934.997783  783.189850  796.698220  787.169836   

   model2_BIC  model3_AIC  model3_BIC  model4_AIC  ...  \
0  490.935836  459.024671  486.041412  465.552321  ...   
1  515.969441  493.004856  520.021596  525.775421  ...   
2  809.683787  785.846002  812.862742  784.122801  ...   

   delta_AIC_model2_vs_model1  delta_AIC_model3_vs_model0  \
0                   -6.082812                 -419.671983   
1                  -31.987468                 -428.519576   
2                    3.979986                 -122.135041   

   delta_AIC_model3_vs_model1  delta_AIC_model3_vs_model2  \
0                  -15.480027                   -9.397215   
1                  -32.438103                   -0.450635   
2                    2.656152             